## Project Description: Next Word Prediction Using LSTM

This project predicts the next word in a sequence using deep learning.

Pipeline:

$$
\text{Text} \rightarrow \text{Tokenization} \rightarrow \text{Sequences} \rightarrow \text{LSTM/GRU} \rightarrow \text{Next Word}
$$

## Step 1: Data Collection

We use Shakespeare's Hamlet text.

$$
\text{Dataset} = \text{Raw Text Corpus}
$$

In [71]:
## Data Collection
# Import NLTK library for natural language processing
import nltk

# Download the gutenberg corpus (collection of classic literature texts)
nltk.download('gutenberg')
# Import the gutenberg corpus module
from nltk.corpus import gutenberg
# Import pandas for data manipulation (though not heavily used in this script)
import  pandas as pd

## Load Shakespeare's Hamlet text data
# Re-import required modules
import nltk
nltk.download('gutenberg')
# Retrieve the raw text of Shakespeare's Hamlet
data=gutenberg.raw('shakespeare-hamlet.txt')
# Re-import necessary modules
from nltk.corpus import gutenberg
import  pandas as pd

## Save the raw text to a local file for future use
# Open the file in write mode and save the hamlet text
with open('data/hamlet.txt','w') as file:
    file.write(data)


[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/psundara/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/psundara/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


## Step 2: Tokenization

Convert words into integers:

$$
\text{word} \rightarrow \text{index}
$$

Vocabulary size:

$$
V = |\text{unique words}|
$$

In [72]:
## Data Preprocessing

# Import necessary libraries for data processing
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Load the hamlet text file and convert to lowercase for consistency
with open('data/hamlet.txt', 'r') as file:
    text = file.read().lower()

# Create a Tokenizer object - converts words into numerical sequences
tokenizer = Tokenizer()
# Fit the tokenizer on the entire text corpus to build vocabulary
tokenizer.fit_on_texts([text])
# Get the total number of unique words (+1 for reserved index 0)
total_words = len(tokenizer.word_index) + 1

# Print the vocabulary size
print(total_words)

4818


## Step 3: Create Input Sequences

We generate n-grams:

$$
[w_1] \rightarrow [w_1, w_2] \rightarrow [w_1, w_2, w_3]
$$

This helps model learn context.

In [73]:
## Create input sequences

# Initialize an empty list to store all generated n-gram sequences
input_sequences = []

# Iterate through each line in the text
for line in text.split('\n'):
    # Convert the line text into a sequence of token indices
    token_list = tokenizer.texts_to_sequences([line])[0]
    # Generate n-grams: progressively longer sequences starting from 1 word
    for i in range(1, len(token_list)):
        # Create n-gram: [w_1], [w_1, w_2], [w_1, w_2, w_3], etc.
        n_gram_sequence = token_list[: i + 1]
        # Add the n-gram sequence to our list
        input_sequences.append(n_gram_sequence)

In [74]:
input_sequences

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891],
 [57, 407],
 [57, 407, 2],
 [57, 407, 2, 1181],
 [57, 407, 2, 1181, 177],
 [57, 407, 2, 1181, 177, 1892],
 [407, 1182],
 [407, 1182, 63],
 [408, 162],
 [408, 162, 377],
 [408, 162, 377, 21],
 [408, 162, 377, 21, 247],
 [408, 162, 377, 21, 247, 882],
 [18, 66],
 [451, 224],
 [451, 224, 248],
 [451, 224, 248, 1],
 [451, 224, 248, 1, 30],
 [408, 407],
 [451, 25],
 [408, 6],
 [408, 6, 43],
 [408, 6, 43, 62],
 [408, 6, 43, 62, 1893],
 [408, 6, 43, 62, 1893, 96],
 [408, 6, 43, 62, 1893, 96, 18],
 [408, 6, 43, 62, 1893, 96, 18, 566],
 [451, 71],
 [451, 71, 51],
 [451, 71, 51, 1894],
 [451, 71, 51, 1894, 567],
 [451, 71, 51, 1894, 567, 378],
 [451, 71, 51, 1894, 567, 378, 80],
 [451, 71, 51, 1894, 567, 378, 80, 3],
 [451, 71, 51, 1894, 567, 378, 80, 3, 273],
 [451, 71

## Step 4: Padding

All sequences must have equal length:

$$
\text{max length} = L
$$

In [75]:
## Pad Sequences
max_sequence_len = max([len(x) for x in input_sequences])
max_sequence_len

14

In [76]:
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
input_sequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]],
      shape=(25732, 14), dtype=int32)

## Step 5: Split Features and Labels

We split:

$$
X = \text{sequence without last word}
$$

$$
y = \text{last word}
$$

In [77]:
##create predictors and label - splitting sequences into input features and target output

# Import TensorFlow library for deep learning utilities
import tensorflow as tf

# Split input sequences into features (X) and labels (y) for supervised learning
# X = all elements except the last one: [w_1, w_2, ...., w_n-1] (input context)
# y = all elements except the first one: [w_2, w_3, ...., w_n] (next word to predict)
x, y = input_sequences[:, :-1], input_sequences[:, -1]

In [78]:
x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]],
      shape=(25732, 13), dtype=int32)

In [79]:
y

array([ 687,    4,   45, ..., 1047,    4,  193],
      shape=(25732,), dtype=int32)

## Step 6: One-Hot Encoding

Convert labels to categorical:

$$
y \rightarrow \text{one-hot vector}
$$

In [80]:
# Convert the labels (y) into one-hot encoded categorical format
# One-hot encoding: each word index becomes a binary vector where only one element is 1
# Example: index 5 becomes [0,0,0,0,0,1,0,...,0] for num_classes dimensions
# This is required for neural network classification tasks
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Display the shape and content of the one-hot encoded labels
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(25732, 4818))

## Step 7: Train-Test Split

Split dataset:

$$
\text{Train} = 80\%, \quad \text{Test} = 20\%
$$

In [81]:
## Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## Step 8: Early Stopping

Stops training when validation loss stops improving.

In [87]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

## Step 9: LSTM & GRU Model

Architecture:

$$
\text{Embedding} \rightarrow \text{LSTM} \rightarrow \text{LSTM} \rightarrow \text{Dense}
$$

In [106]:
## -------------------------------------------------------------
## LSTM-based Text Generation Model
## -------------------------------------------------------------
## This model is designed to learn sequential patterns in text
## data and predict the next word in a sequence.
##
## Architecture Overview:
##
## 1. Embedding Layer:
##    - Input: Integer-encoded words (tokenized text)
##    - Output: Dense vector representation of words
##    - Purpose:
##        Converts sparse word indices into dense vectors
##        where semantic relationships between words can be learned.
##    - Input Dim  : total_words (size of vocabulary)
##    - Output Dim : 100 (embedding vector size)
##
## 2. First LSTM Layer:
##    - Units: 150
##    - return_sequences=True
##    - Purpose:
##        Captures sequential dependencies between words.
##        return_sequences=True ensures full sequence output is passed
##        to the next LSTM layer (stacked LSTM architecture).
##
## 3. Dropout Layer:
##    - Rate: 0.2
##    - Purpose:
##        Prevents overfitting by randomly dropping 20% of neurons
##        during training.
##
## 4. Second LSTM Layer:
##    - Units: 100
##    - Purpose:
##        Further refines sequence understanding and outputs a
##        single vector representing the sequence context.
##
## 5. Dense Output Layer:
##    - Units: total_words
##    - Activation: softmax
##    - Purpose:
##        Outputs probability distribution over entire vocabulary.
##        Each neuron represents the likelihood of the next word.
##
## Model Configuration:
##
## - Loss Function:
##     categorical_crossentropy
##     → Used because this is a multi-class classification problem
##       (predicting one word out of total_words).
##
## - Optimizer:
##     Adam
##     → Adaptive optimizer for faster convergence.
##
## - Metrics:
##     Accuracy
##     → Measures how often the predicted word matches the actual word.
##
## Input Shape:
##    (None, max_sequence_len)
##    - None → batch size (dynamic)
##    - max_sequence_len → length of input sequences
##
## Overall Goal:
##    Given a sequence of words, predict the most probable next word.
##    This is commonly used for:
##        - Text generation
##        - Language modeling
##        - Autocomplete systems
##
## -------------------------------------------------------------

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Dense,
    LSTM,
    Dropout,
    GRU
)

# Define the model
model = Sequential()
model.add(Embedding(total_words, 100))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.build(input_shape=(None, max_sequence_len - 1))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ (None, 13, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [107]:
# Define the model
gru_model = Sequential()
gru_model.add(Embedding(total_words, 100))
gru_model.add(GRU(150, return_sequences=True))
gru_model.add(Dropout(0.2))
gru_model.add(GRU(100))
gru_model.add(Dense(total_words, activation='softmax'))

gru_model.build(input_shape=(None, max_sequence_len - 1))
gru_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 13, 150)        │       113,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 100)            │        75,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,157,418 (4.42 MB)

 Trainable params: 1,157,418 (4.42 MB)

 Non-trainable params: 0 (0.00 B)

## Step 10: Training

Model learns:

$$
P(w_t | w_1, w_2, ..., w_{t-1})
$$

In [ ]:
## -------------------------------------------------------------
## Model Training
## -------------------------------------------------------------
## This step trains the LSTM model on the prepared dataset.
##
## model.fit(...) is the core training loop where:
##   - The model learns patterns from input sequences (X_train)
##   - Adjusts weights using backpropagation
##   - Minimizes the loss function over multiple iterations
##
## Parameters Explained:
##
## 1. X_train:
##    - Input sequences (tokenized text)
##    - Shape: (num_samples, sequence_length)
##
## 2. y_train:
##    - Target labels (next word prediction)
##    - One-hot encoded vectors of size = total_words
##
## 3. epochs = 50:
##    - An epoch means one complete pass over the entire training dataset.
##
##    👉 Example:
##       If you have 10,000 samples:
##       - 1 epoch = model sees all 10,000 samples once
##       - 50 epochs = model sees the data 50 times
##
##    👉 Why increasing epochs improves accuracy:
##       - Initially, model weights are random → poor predictions
##       - With each epoch:
##           • Model learns patterns in data
##           • Loss decreases
##           • Accuracy improves
##
##    👉 BUT:
##       - Too few epochs → underfitting (model doesn't learn enough)
##       - Too many epochs → overfitting (model memorizes data, poor generalization)
##
##    👉 Ideal approach:
##       Train until validation accuracy stabilizes (use EarlyStopping in real projects)
##
## 4. batch_size = 32:
##    - Number of samples processed before updating model weights
##
##    👉 Smaller batch:
##       - More updates, slower but better generalization
##
##    👉 Larger batch:
##       - Faster training, but may generalize poorly
##
## 5. validation_data = (X_test, y_test):
##    - Used to evaluate model performance on unseen data after each epoch
##    - Helps detect overfitting
##
## 6. verbose = 1:
##    - Displays progress bar during training
##
## Output:
##    - history object stores:
##        • training loss & accuracy
##        • validation loss & accuracy
##    - Useful for plotting learning curves
##
## Overall Goal:
##    Train the model to accurately predict the next word in a sequence
##    while generalizing well to unseen text.
##
## -------------------------------------------------------------

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

In [108]:
## Train the model
gru_history=gru_model.fit(X_train,y_train,epochs=50,validation_data=(X_test,y_test),verbose=1)

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.0321 - loss: 6.9877 - val_accuracy: 0.0332 - val_loss: 6.7391
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.0447 - loss: 6.4467 - val_accuracy: 0.0470 - val_loss: 6.7125
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.0569 - loss: 6.1756 - val_accuracy: 0.0618 - val_loss: 6.6667
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.0689 - loss: 5.8967 - val_accuracy: 0.0664 - val_loss: 6.7270
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.0831 - loss: 5.5991 - val_accuracy: 0.0738 - val_loss: 6.8069
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.0956 - loss: 5.3023 - val_accuracy: 0.0734 - val_loss: 6.9043
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.1095 - loss: 5.0219 - val_accuracy: 0.0686 - val_loss: 6.9891
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 13s 19ms/step - accuracy: 0.1233 - loss: 4.7604 - 

## Step 11: Next Word Prediction

Given input:

$$
w_1, w_2, ..., w_n \rightarrow w_{n+1}
$$

In [ ]:
def predict_next_word(model, tokenizer, text, max_sequence_len):

    # text → numbers
    token_list = tokenizer.texts_to_sequences([text])[0]

    # keep last words only
    token_list = token_list[-(max_sequence_len - 1):]

    # make fixed length
    token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')

    # model prediction
    predicted_index = np.argmax(model.predict(token_list, verbose=0))

    # directly get word (no loop!)
    return tokenizer.index_word.get(predicted_index, "?")

## ✨ Next Word Prediction Demo

We test the model with a meaningful Shakespeare-style sentence:

$$
\text{Input} \rightarrow \text{Model} \rightarrow \text{Predicted Next Word}
$$

In [109]:
input_text = "To be or not to be"

# Meaningful input
#input_text = "the king is"

# Get max sequence length
max_sequence_len = gru_model.input_shape[1] + 1

# Predict next word
next_word = predict_next_word(gru_model, tokenizer, input_text, max_sequence_len)

# Beautified output
print("\n" + "="*50)
print("🔮 NEXT WORD PREDICTION")
print("="*50)

print(f"📝 Input Sentence : {input_text}")
print(f"✨ Predicted Word : {next_word}")

print("\n📢 Complete Sentence:")
print(f"👉 {input_text} {next_word}")

print("="*50 + "\n")


🔮 NEXT WORD PREDICTION
📝 Input Sentence : To be or not to be
✨ Predicted Word : an

📢 Complete Sentence:
👉 To be or not to be an



## Step 12: Save Model

Save trained model and tokenizer for reuse.

In [110]:
gru_model.save('model/gru_next_word_lstm.keras')

# Save tokenizer

tokenizer_json = tokenizer.to_json()
with open('model/ntokenizer.json', 'w') as file:
    file.write(tokenizer_json)

In [113]:
inputs = ["Barn. Last night of all,When yond same", "Enter Barnardo and Francisco two", "He may approue our eyes, and speake to", "The Bell then beating"]

for input_text in inputs:

    # Meaningful input
    #input_text = "the king is"

    # Get max sequence length
    max_sequence_len = gru_model.input_shape[1] + 1

    # Predict next word
    next_word = predict_next_word(gru_model, tokenizer, input_text, max_sequence_len)

    # Beautified output
    print("\n" + "="*50)
    print("🔮 NEXT WORD PREDICTION")
    print("="*50)

    print(f"📝 Input Sentence : {input_text}")
    print(f"✨ Predicted Word : {next_word}")

    print("\n📢 Complete Sentence:")
    print(f"👉 {input_text} {next_word}")

    print("="*50 + "\n")



🔮 NEXT WORD PREDICTION
📝 Input Sentence : Barn. Last night of all,When yond same
✨ Predicted Word : fortune

📢 Complete Sentence:
👉 Barn. Last night of all,When yond same fortune


🔮 NEXT WORD PREDICTION
📝 Input Sentence : Enter Barnardo and Francisco two
✨ Predicted Word : centinels

📢 Complete Sentence:
👉 Enter Barnardo and Francisco two centinels


🔮 NEXT WORD PREDICTION
📝 Input Sentence : He may approue our eyes, and speake to
✨ Predicted Word : him

📢 Complete Sentence:
👉 He may approue our eyes, and speake to him


🔮 NEXT WORD PREDICTION
📝 Input Sentence : The Bell then beating
✨ Predicted Word : one

📢 Complete Sentence:
👉 The Bell then beating one

